In [1]:
import requests
import pandas as pd
import json
import time
from config import api_key
from IPython.display import display

In [2]:
valid_shards = ["steam", "console"]

### Steam Data Extraction

In [3]:
##Fetches a list of recent match IDs from the PUBG API (It is the only way to get a random, high-volume list of matches without knowing specific players beforehand)

def get_pubg_samples_steam(api_key, shard=valid_shards[0]):
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [4]:
match_samples = get_pubg_samples_steam(api_key)

print(match_samples)


{'data': {'type': 'sample', 'id': '51403435-e829-484a-a194-58169b435762', 'attributes': {'createdAt': '2026-05-10T00:00:00Z', 'titleId': 'bluehole-pubg', 'shardId': 'steam'}, 'relationships': {'matches': {'data': [{'type': 'match', 'id': '5cc9dd84-d35a-4fd8-8ab4-7c77f7543722'}, {'type': 'match', 'id': '1e502158-23c3-41ff-abf0-f5801c820f1d'}, {'type': 'match', 'id': '11244946-567b-4630-82c4-43ea21afe159'}, {'type': 'match', 'id': 'd098e207-ac97-4837-a54d-42c2e72f5356'}, {'type': 'match', 'id': '95b1c874-de3c-4df8-82e2-665dbfe372a4'}, {'type': 'match', 'id': 'e3737d41-4fa2-4483-a3f8-84c4a4fb4ce5'}, {'type': 'match', 'id': '5ac55520-4224-4d07-bba3-2ed65ab0864c'}, {'type': 'match', 'id': '5845b017-c877-4a02-b119-ccf921a89733'}, {'type': 'match', 'id': 'bfd90f35-802f-42e6-83ba-023f6f63811b'}, {'type': 'match', 'id': 'af45f2c6-455e-4ad2-bd4a-1248c1af2560'}, {'type': 'match', 'id': '0c28f018-c84a-47cf-9830-63996499a959'}, {'type': 'match', 'id': '48644a18-f59e-4681-9399-4dc5e179bc01'}, {'type

In [5]:
matches_list = match_samples['data']['relationships']['matches']['data']
match_samples_df = pd.DataFrame(matches_list)
match_samples_df.rename(columns={'id': 'match_id'}, inplace=True)
print(f"Total matches retrieved: {len(match_samples_df)}")
display(match_samples_df.head())

Total matches retrieved: 1339


,type,match_id
0,match,5cc9dd84-d35a-4fd8-8ab4-7c77f7543722
1,match,1e502158-23c3-41ff-abf0-f5801c820f1d
2,match,11244946-567b-4630-82c4-43ea21afe159
3,match,d098e207-ac97-4837-a54d-42c2e72f5356
4,match,95b1c874-de3c-4df8-82e2-665dbfe372a4


In [6]:
match_id = match_samples_df['match_id'].tolist()

In [7]:
def get_match_details(api_key, match_id, shard=valid_shards[0]):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [8]:
match_details = get_match_details(api_key,match_id[0])
print(match_details)

{'data': {'type': 'match', 'id': '5cc9dd84-d35a-4fd8-8ab4-7c77f7543722', 'attributes': {'createdAt': '2026-05-09T23:55:39Z', 'duration': 204, 'gameMode': 'solo', 'tags': None, 'matchType': 'tutorialatoz', 'seasonState': 'progress', 'stats': None, 'titleId': 'bluehole-pubg', 'shardId': 'steam', 'mapName': 'Tutorial_Main', 'isCustomMatch': False}, 'relationships': {'assets': {'data': [{'type': 'asset', 'id': '2dfb470b-4c03-11f1-ab28-f6c712bff5da'}]}, 'rosters': {'data': [{'type': 'roster', 'id': '817460cb-a1d0-43d4-82e6-99e6b394b560'}]}}, 'links': {'self': 'https://api.pubg.com/shards/steam/matches/5cc9dd84-d35a-4fd8-8ab4-7c77f7543722', 'schema': ''}}, 'included': [{'type': 'participant', 'id': 'a5f825b0-8c87-4a62-8f0d-924ac4e7e520', 'attributes': {'stats': {'DBNOs': 0, 'assists': 0, 'boosts': 2, 'damageDealt': 0, 'deathType': 'alive', 'headshotKills': 0, 'heals': 1, 'killPlace': 1, 'killStreaks': 0, 'kills': 0, 'longestKill': 0, 'name': 'BRoma17', 'playerId': 'account.644fe95b5e234c23ba

In [9]:
match_details = get_match_details(api_key,match_id[0])
matches_details_list = match_details['data']['relationships']['assets']['data']
print(matches_details_list)

[{'type': 'asset', 'id': '2dfb470b-4c03-11f1-ab28-f6c712bff5da'}]


In [10]:
def extract_telemetry_url(match_details):
    included_items = match_details.get("included", [])
    
    for item in included_items:
        is_telemetry = item.get("attributes", {}).get("name") == "telemetry"
        
        if is_telemetry:
            telemetry_url = item["attributes"]["URL"]
            print("Telemtry URL sucessfully extracted!")
            return telemetry_url
            
    print("Did not find telemetry URL in match details:(")
    return None

In [12]:
telemetry_url = extract_telemetry_url(match_details)
print(telemetry_url)
# Save the URL to Jupyter's internal storage so it can be accessed in other notebooks
%store telemetry_url


Telemtry URL sucessfully extracted!
https://telemetry-cdn.pubg.com/bluehole-pubg/steam/2026/05/09/23/59/2dfb470b-4c03-11f1-ab28-f6c712bff5da-telemetry.json
Stored 'telemetry_url' (str)
